# 01 - data acquisition

**purpose**: explore ravdess audio files, create speaker-disjoint train/val/test split, verify integrity, and export label files for downstream notebooks.

**input**: `data/raw/` (original zenodo download structure)

**output**: `data/{train,val,test}/{speech,song}/Actor_XX/*.wav` + `data/processed/split_labels.csv`

In [9]:
import shutil
from pathlib import Path
import pandas as pd

import sys
sys.path.append('../../')
from src.config.settings import (
    RAW_SPEECH_DIR,
    RAW_SONG_DIR,
    DATA_DIR,
    PROCESSED_DIR,
    LABELS_FILE,
)
from src.utils.helpers import count_wavs, collect_wavs, assign_split

## 1. raw data structure

we only use audio-only files (modality=03). video files are ignored.

speech wavs

In [13]:
print(f"speech wavs: {count_wavs(RAW_SPEECH_DIR)}")

speech wavs: 1440


song wavs

In [12]:
print(f"song wavs: {count_wavs(RAW_SONG_DIR)}")

song wavs: 1012


In [14]:
print(f"total speech and song wavs: {count_wavs(RAW_SPEECH_DIR) + count_wavs(RAW_SONG_DIR)}")

total speech and song wavs: 2452


## 2. parse filenames into metadata

filename format: `03-CHANNEL-EMOTION-INTENSITY-STATEMENT-REPETITION-ACTOR.wav`

see `src.utils.helpers.parse_filename` and `src.config.settings.EMOTION_MAP` for details.

In [15]:
df_speech = collect_wavs(RAW_SPEECH_DIR)
df_speech.head()

,filepath,channel,emotion_code,emotion,intensity,statement,repetition,actor,gender
0,/mnt/d/career/projects/lightweight-speech-emot...,speech,1,neutral,normal,1,1,1,male
1,/mnt/d/career/projects/lightweight-speech-emot...,speech,1,neutral,normal,1,2,1,male
2,/mnt/d/career/projects/lightweight-speech-emot...,speech,1,neutral,normal,2,1,1,male
3,/mnt/d/career/projects/lightweight-speech-emot...,speech,1,neutral,normal,2,2,1,male
4,/mnt/d/career/projects/lightweight-speech-emot...,speech,2,calm,normal,1,1,1,male


In [16]:
df_song = collect_wavs(RAW_SONG_DIR)
df_song.head()

,filepath,channel,emotion_code,emotion,intensity,statement,repetition,actor,gender
0,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,1,1,1,male
1,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,1,2,1,male
2,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,2,1,1,male
3,/mnt/d/career/projects/lightweight-speech-emot...,song,1,neutral,normal,2,2,1,male
4,/mnt/d/career/projects/lightweight-speech-emot...,song,2,calm,normal,1,1,1,male


In [17]:
df_all = pd.concat([df_speech, df_song], ignore_index=True)
print(f"parsed: {len(df_speech)} speech + {len(df_song)} song = {len(df_all)} total")

parsed: 1440 speech + 1012 song = 2452 total


emotion distribution:

In [18]:
print(df_all["emotion"].value_counts().sort_index())


emotion
angry        376
calm         376
disgust      192
fearful      376
happy        376
neutral      188
sad          376
surprised    192
Name: count, dtype: int64


In [20]:
print(f"actors: {sorted(df_all['actor'].unique())}")


actors: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


In [21]:
print(f"gender split: {df_all.groupby('gender')['actor'].nunique().to_dict()}")

gender split: {'female': 12, 'male': 12}


## 3. speaker-disjoint split

by actor id (not random files) to test new speakers.

| split | actors | speech | song |
|-------|--------|--------|------|
| train | 01-19 | 19 actors | 18 actors (no actor 18 in song) |
| val   | 20-22 | 3 actors  | 3 actors |
| test  | 23-24 | 2 actors  | 2 actors |

In [22]:
df_all["split"] = df_all["actor"].apply(assign_split)

pivot = df_all.groupby(["split", "channel"]).size().unstack(fill_value=0)
pivot["total"] = pivot.sum(axis=1)
print(pivot)

channel  song  speech  total
split                       
test       88     120    208
train     792    1140   1932
val       132     180    312


## 4. copy to split directories

destination: `data/{train,val,test}/{speech,song}/Actor_XX/`

In [23]:
if count_wavs(RAW_SPEECH_DIR) > 0:
    for _, row in df_all.iterrows():
        src = Path(row["filepath"])
        dest = DATA_DIR / row["split"] / row["channel"] / f"Actor_{row['actor']:02d}" / src.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(str(src), str(dest))
    print("files copied to split directories")
else:
    print("files already copied to split directories, skipping copy step")

files copied to split directories


In [24]:
def _get_dest(row):
    return str(DATA_DIR / row["split"] / row["channel"] / f"Actor_{row['actor']:02d}" / Path(row["filepath"]).name)


In [25]:
df_all["filepath"] = df_all.apply(_get_dest, axis=1)
all_ok = all(Path(p).exists() for p in df_all["filepath"])
print(f"all {len(df_all)} files verified at destination: {all_ok}")

all 2452 files verified at destination: True


In [26]:
actor_splits = df_all.groupby("actor")["split"].unique()
assert all(len(v) == 1 for v in actor_splits), "actor leak detected!"
print("no actor leaks between splits.")


no actor leaks between splits.


In [27]:

gender_pivot = df_all.groupby(["split", "gender"]).size().unstack(fill_value=0)
print("\ngender balance:")
print(gender_pivot)


gender balance:
gender  female  male
split               
test       104   104
train      892  1040
val        208   104


## 5. export labels for downstream

save `split_labels.csv` to `data/processed/`. downstream notebooks load this csv.

In [28]:
EXPORT_COLS = ["filepath", "split", "channel", "emotion_code", "emotion",
              "intensity", "actor", "gender", "statement", "repetition"]


In [29]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_all[EXPORT_COLS].to_csv(LABELS_FILE, index=False)
print(f"saved {len(df_all)} labels to {LABELS_FILE}")

saved 2452 labels to /mnt/d/career/projects/lightweight-speech-emotion-recognition-on-open-datasets/data/processed/split_labels.csv


In [34]:
check = pd.read_csv(LABELS_FILE)
print(check.head(3))

                                            filepath  split channel  \
0  /mnt/d/career/projects/lightweight-speech-emot...  train  speech   
1  /mnt/d/career/projects/lightweight-speech-emot...  train  speech   
2  /mnt/d/career/projects/lightweight-speech-emot...  train  speech   

   emotion_code  emotion intensity  actor gender  statement  repetition  
0             1  neutral    normal      1   male          1           1  
1             1  neutral    normal      1   male          1           2  
2             1  neutral    normal      1   male          2           1  


In [35]:
print(f"shape: {check.shape}")
print(f"splits: {check['split'].value_counts().to_dict()}")
print(f"channels: {check['channel'].value_counts().to_dict()}")

shape: (2452, 10)
splits: {'train': 1932, 'val': 312, 'test': 208}
channels: {'speech': 1440, 'song': 1012}


## summary

- **total files**: 2452 (1440 speech + 1012 song)
- **split**: train actors 01-19, val 20-22, test 23-24
- **integrity**: no actor leaks, gender balanced
- **labels**: `data/processed/split_labels.csv` ready for eda
- **next**: `02-eda` notebook